In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import scanpy as sc
import squidpy as sq
import seaborn as sns

In [2]:
if not os.path.exists(f'../../results/03_eval_simulate_data/04_sparsity'):
    os.makedirs('../../results/03_eval_simulate_data/04_sparsity', exist_ok=True)

In [3]:
dataset_list = ['01_10x_Visium_mouse_brain',
                '02_10x_Visium_Human_Heart_MI',
                '03_10x_Visium_Human_Colorectal_Cancer',
                '04_10x_Visium_Human_Heart',
                '05_10x_Visium_Mouse_Embryo',
                '06_10x_Visium_Mouse_Olfactory_Bulb',
                '07_10x_Visium_Human_Breast_Cancer',
                '08_10x_Visium_Human_Lymph_Node',
                '09_10x_Visium_Human_Prostate',
                '10_10x_Visium_Human_Prostate_Cancer',
                '11_10x_Visium_Human_Cerebellum',
                '12_10x_Visium_Mouse_Kidney',
                '13_10x_Visium_Human_Lung_Cancer',
                '14_10x_Visium_Human_Brain_Cancer',
                '15_10x_Visium_Human_Kidney',
                '16_10x_Visium_Human_Intestine_Cancer',
                '17_10x_Visium_Human_Skin_Melanoma',
                '18_10x_Visium_Human_Cervical_Cancer',
                '19_10x_Visium_Human_Breast_Cancer',
                '20_10x_Visium_Human_Heart_MI',
               ]

In [5]:
df_res = pd.read_csv('../../results/03_eval_simulate_data/03_compute_correlation_per_gene.csv')

In [6]:
for dataset in dataset_list:
    adata = sc.read_h5ad(f"../../results/01_generate_simulate_data/{dataset}.h5ad")
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    
    filtered_gene_names = [gene for gene in adata.var.index if '_1' in gene]
    
    # Subset 'adata' to keep only the filtered genes
    adata_filtered = adata[:, filtered_gene_names]
    
    sc.pp.calculate_qc_metrics(adata_filtered, percent_top=(10, ), inplace=True)
    df = adata_filtered.var.set_index('gene')
    
    df_res2 = df_res[df_res['dataset']== dataset]
    df_res2 = df_res2[['gene', 'method', 'correlation']]
    df_res2 = df_res2.pivot_table(values="correlation", index="gene", columns="method")
    
    df = pd.concat([df, df_res2], axis=1)
    
    df = df.drop(['spatial_var', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 
              'total_counts', 'log1p_total_counts'], axis=1)
    
    df_corr = df.corr()[['n_cells_by_counts']].drop(['n_cells_by_counts'])
    
    df_corr = df_corr.reset_index()
    df_corr.columns = ['method', 'n_cells_by_counts']
    
    df_corr.to_csv(f'../../results/03_eval_simulate_data/04_sparsity/{dataset}_corr.csv')
    df.to_csv(f'../../results/03_eval_simulate_data/04_sparsity/{dataset}.csv')

In [ ]:
df_corr = df.corr()[['n_cells_by_counts']].drop(['n_cells_by_counts'])

In [ ]:
df_corr = df_corr.reset_index()
df_corr.columns = ['method', 'n_cells_by_counts']

In [ ]:
df_corr

In [ ]:
ax = sns.barplot(df_corr, x='method', y='n_cells_by_counts')
ax.tick_params(axis='x', rotation=60)
sns.despine(trim=True)

In [ ]:
sns.regplot(df, x='n_cells_by_counts', y='MoranI')